# Spatial discretization
\
FV and DG

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive
import ipywidgets as widgets
import os

from cadet import Cadet
import utility.convergence as convergence
import utility.setting_Col1D_SMA_4comp_LWE_benchmark1 as lwe

Cadet.cadet_path = r"C:\Users\jmbr\OneDrive\Desktop\CADET_compiled\master5_generalizedUnit_f1a1972\aRELEASE"
path = os.getcwd()

def graph_column(spatial_method=3, nAxElem=2, nParElem=1):

    axRefinement = nAxElem / 8
    model = Cadet()
    model.root = lwe.get_model(
        spatial_method_bulk=spatial_method,
        spatial_method_particle=spatial_method,
        particle_type='GENERAL_RATE_PARTICLE',
        axRefinement=axRefinement,
        parZ=nParElem,
        return_bulk=True,
        idas_abstol=1E-8,
        idas_reltol=1E-6
    )
    
    model.filename = 'test.h5'
    model.save()
    model.run_simulation()

    outlet = convergence.get_outlet(path+'/'+model.filename, unit="000")
    sol_time = convergence.get_solution_times(path+'/'+model.filename)
    
    reference = convergence.get_outlet(path+'/data/ref_LWE.h5', unit="000")
    errorComp = np.max(abs(reference[:, 1:] - outlet[:, 1:])) / np.max(abs(reference[:, 1:]))
    errorSalt = np.max(abs(reference[:, 0] - outlet[:, 0])) / np.max(abs(reference[:, 0]))
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    
    # First plot
    axs[0].plot(sol_time, outlet[:, 0])
    axs[0].set_xlabel(r'$x~/~M$')
    axs[0].set_ylabel(r'$concentration~/~mol \cdot M^{-3}$')
    axs[0].set_title("Salt")
    
    axs[1].plot(sol_time, outlet[:, 1:])
    axs[1].set_xlabel(r'$x~/~M$')
    axs[1].set_title("components")

    fig.suptitle(
        f"Sim. time: {convergence.get_compute_time(path + '/' + model.filename):.3e}" + f", rel. max. error salt: {errorSalt*100:.2f}%," + f" rel. max. error comps: {errorComp*100:.2f}%",
        fontsize=14
    )
    
    plt.tight_layout()
    plt.show()

spatial_method_options = [0, 1, 2, 3, 4, 5]
nAxElem_options = [4, 8, 16, 32, 64]
nParElem_options = [1, 2, 4, 8, 16]

interact(
    graph_column,
    spatial_method=widgets.SelectionSlider(
        options=spatial_method_options,
        description="Method"
    ),
    nAxElem=widgets.SelectionSlider(
        options=nAxElem_options,
        description="Axial cells"
    ),
    nParElem=widgets.SelectionSlider(
        options=nParElem_options,
        description="Particle cells"
    )
)

